# 06 — Story EDA: Finding the Four Chapters

## Purpose
Run the actual exploratory analysis to discover and validate the narrative.
Each section answers one story question and generates a chart for the dashboard.

**This notebook decides the final story — run it after all pipeline notebooks (01–05).**

## Inputs
- `data/clean/master_sa3.csv` — main joined dataset (SA3 × year)
- `data/clean/stars_timeline.csv` — facility-level quality across 12 snapshots
- `data/clean/demographics_acpr.csv` — Indigenous / CALD / age (ACPR level)

## Story structure: "The Unequal Race"
| Chapter | Question | Key data |
|---------|----------|----------|
| 1 | Did the Oct 2023 reform actually improve quality? Equally? | stars_timeline by MMM |
| 2 | Which regions are in a supply-demand collision zone? | master_sa3 access + supply |
| 3 | Who is being left behind (demographics)? | demographics_acpr |
| 4 | The reveal: which SA3s are worst on every dimension? | master_sa3 care_gap_index |

## Narrative target (fill in after running)
After this notebook you should be able to complete:
> "The Oct 2023 staffing mandate improved quality in [metro/remote/both], but [region X]
> remains critically underserved because [supply/quality/access reason].
> The people most at risk are [demographic group], concentrated in [geography]."

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import os

CLEAN  = '../../data/clean'
ASSETS = '../../assets'
os.makedirs(ASSETS, exist_ok=True)

master = pd.read_csv(f'{CLEAN}/master_sa3.csv')
stars  = pd.read_csv(f'{CLEAN}/stars_timeline.csv')
stars['snapshot_date'] = pd.to_datetime(stars['snapshot_date'])

# Feb 2026 star ratings exist but access/supply/pop only go to 2025.
# Use 2025 as the "latest complete" year for all current-state charts.
LATEST_YEAR = int(master[master['care_gap_index'].notna()]['year'].max())

print('master:', master.shape, '| years:', sorted(master['year'].unique()))
print('stars: ', stars.shape,  '| snapshots:', stars['snapshot'].nunique())
print(f'Latest complete year: {LATEST_YEAR}')

MMM_LABELS = {
    'MM1': 'Metro', 'MM2': 'Regional centre', 'MM3': 'Large rural',
    'MM4': 'Medium rural', 'MM5': 'Small rural', 'MM6': 'Remote', 'MM7': 'Very remote'
}
MMM_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']

## Chapter 1 — Did the Reform Work?

**Question:** Did the Oct 2023 staffing mandate (400 care mins/day, 40 RN mins/day)
actually lift quality scores? And was the improvement equal across remoteness classes?

**What to look for:**
- All MMM classes rising after Oct 2023 → reform worked broadly
- Remote (MM6/MM7) catching up to metro → reform reduced the gap
- Remote flatlines or drops → mandate imposed costs without capacity to comply
- Any MMM class declining after mandate → possible facility closures masking improvement

In [ ]:
# =============================================================================
# CHART 1: Quality score trend by remoteness class (all 12 snapshots)
# =============================================================================

trend = (
    stars.dropna(subset=['mmm_code', 'quality_score'])
    .groupby(['snapshot_date', 'mmm_code'])['quality_score']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
for i, mmm in enumerate(['MM1', 'MM2', 'MM3', 'MM4', 'MM5', 'MM6', 'MM7']):
    sub = trend[trend['mmm_code'] == mmm].sort_values('snapshot_date')
    if len(sub) == 0:
        continue
    ax.plot(sub['snapshot_date'], sub['quality_score'],
            label=MMM_LABELS.get(mmm, mmm), color=MMM_COLORS[i],
            marker='o', markersize=4, linewidth=2)

ax.axvline(pd.Timestamp('2023-10-01'), color='red', linestyle='--',
           linewidth=1.5, alpha=0.8, label='Mandate Oct 2023')
ax.set_title('Quality Score Trend by Remoteness Class (May 2023 → Feb 2026)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Avg Quality Score (1–5 scale)', fontsize=11)
ax.set_xlabel('')
ax.set_ylim(2.5, 5)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{ASSETS}/ch1_quality_trend.png', dpi=150, bbox_inches='tight')
plt.show()

# Quantify the pre→post mandate change
pre  = stars[stars['snapshot'] == 'August 2023'].groupby('mmm_code')['quality_score'].mean()
post = stars[stars['snapshot'] == 'February 2024'].groupby('mmm_code')['quality_score'].mean()
change = pd.DataFrame({'Aug 2023 (pre)': pre, 'Feb 2024 (post)': post})
change['change'] = (change['Feb 2024 (post)'] - change['Aug 2023 (pre)']).round(3)
change['pct_change'] = (change['change'] / change['Aug 2023 (pre)'] * 100).round(1)
print('Quality change by remoteness class (Aug 2023 → Feb 2024):')
print(change.round(3).to_string())
print('\n>> INSIGHT: Fill in after viewing — did remote improve MORE or LESS than metro?')

## Chapter 2 — The Collision Zones

**Question:** Which SA3 regions are caught between rising demand and inadequate supply?

**Two lenses:**
1. **4-quadrant scatter**: home_care_rate vs residential_access_rate — bottom-left quadrant
   (low on both) = access desert; top-right = well-served. Size = pop_65_plus.
2. **Supply pressure**: places_per_1000_elderly — regions below the national median with
   high HCP Level 3/4 share are the "waitlist pressure" zones.

**What to look for:**
- Remote SA3s clustering in the bottom-left (access desert)
- Large circles in the bottom-left = high elderly population with low access = biggest risk
- SA3s where `hcp_high_needs` is high but `n_residential` is low = trapped-at-home signal

In [ ]:
# =============================================================================
# CHART 2A: 4-quadrant scatter — home care vs residential access by SA3
# =============================================================================

latest = master[master['year'] == LATEST_YEAR].copy()

fig, ax = plt.subplots(figsize=(11, 8))

mmm_colors_map = {
    'MM1': '#2196F3', 'MM2': '#4CAF50', 'MM3': '#FF9800',
    'MM4': '#FF5722', 'MM5': '#9C27B0', 'MM6': '#F44336', 'MM7': '#212121'
}

for mmm, grp in latest.dropna(subset=['home_care_rate', 'residential_access_rate']).groupby('mmm_code'):
    size = (grp['pop_65_plus'].fillna(1000) / 150).clip(20, 500)
    ax.scatter(grp['home_care_rate'], grp['residential_access_rate'],
               c=mmm_colors_map.get(mmm, 'gray'),
               label=MMM_LABELS.get(mmm, mmm),
               alpha=0.55, s=size, edgecolors='white', linewidths=0.4)

med_hc  = latest['home_care_rate'].median()
med_res = latest['residential_access_rate'].median()
ax.axvline(med_hc,  color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(med_res, color='gray', linestyle='--', alpha=0.5, linewidth=1)

ax.text(0.02, 0.97, 'Low both\n(access desert)',
        transform=ax.transAxes, va='top', color='red', fontsize=9, alpha=0.8)
ax.text(0.97, 0.97, 'High both\n(well-served)',
        transform=ax.transAxes, va='top', ha='right', color='green', fontsize=9, alpha=0.8)
ax.text(0.97, 0.03, 'High home / Low residential\n(waitlist pressure)',
        transform=ax.transAxes, va='bottom', ha='right', color='orange', fontsize=9, alpha=0.8)

ax.set_xlabel('Home Care Rate (users per 100 pop 65+)', fontsize=11)
ax.set_ylabel('Residential Access Rate (users per 100 pop 65+)', fontsize=11)
ax.set_title(f'Supply-Demand Quadrant by SA3 ({LATEST_YEAR})\n(Circle size = pop 65+)',
             fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(f'{ASSETS}/ch2_4quadrant.png', dpi=150, bbox_inches='tight')
plt.show()

desert = latest[
    (latest['home_care_rate'] < med_hc) &
    (latest['residential_access_rate'] < med_res) &
    latest['pop_65_plus'].notna()
].sort_values('pop_65_plus', ascending=False)

print(f'Access desert SA3s (below median on both): {len(desert)}')
print('\nTop 15 by pop_65_plus (biggest underserved populations):')
print(desert[['sa3_name', 'state', 'mmm_code', 'pop_65_plus',
              'residential_access_rate', 'home_care_rate']].head(15).round(1).to_string(index=False))

In [ ]:
# =============================================================================
# CHART 2B: Worst SA3s on Care Gap Index — ranked bar
# =============================================================================

worst_gap = (
    latest.dropna(subset=['care_gap_index'])
    .sort_values('care_gap_index', ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 7))
colors = [mmm_colors_map.get(m, 'gray') for m in worst_gap['mmm_code']]
bars = ax.barh(worst_gap['sa3_name'], worst_gap['care_gap_index'],
               color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, worst_gap['care_gap_index']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', ha='left', fontsize=8)

ax.set_xlabel('Care Gap Index  (access rate ÷ quality score — higher = worse)', fontsize=10)
ax.set_title(f'Top 20 Most Underserved SA3 Regions — {LATEST_YEAR}',
             fontsize=13, fontweight='bold')
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_handles = [Patch(color=c, label=MMM_LABELS.get(m, m))
                  for m, c in mmm_colors_map.items()
                  if m in worst_gap['mmm_code'].values]
ax.legend(handles=legend_handles, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{ASSETS}/ch2_worst_gap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 20 worst SA3 full detail:')
print(worst_gap[['sa3_name', 'state', 'mmm_code', 'residential_access_rate',
                  'quality_score', 'care_gap_index', 'pop_65_plus', 'n_residential']].round(2).to_string(index=False))

## Chapter 3 — Who Is Being Left Behind?

**Question:** Are Indigenous Australians, CALD communities, and the very elderly
under-represented in care relative to their share of the population?

**What to look for:**
- `pct_indigenous` in residential care vs home care — if home care << residential,
  Indigenous people are accessing the more expensive/intensive stream less
- CALD trend: language barrier may mean delayed entry and higher acuity on admission
- Age at entry trend: if 80–84 is shrinking and 90+ is growing, people are entering
  residential care much later — driven by home care substitution or fear of quality

**Data caveat:** This chapter uses ACPR-level data (73 regions), not SA3.
We can name ACPRs (e.g. "Darwin", "Kimberley") but not drill into specific SA3s.

In [ ]:
# =============================================================================
# CHART 3: Demographics analysis from ACPR-level CURF data
# =============================================================================

demo_path = f'{CLEAN}/demographics_acpr.csv'
if not os.path.exists(demo_path):
    print('Run notebook 04 first to generate demographics_acpr.csv')
else:
    demo = pd.read_csv(demo_path)
    print(f'Demographics shape: {demo.shape}')
    print(f'Years: {sorted(demo["YEAR"].unique())}')

    # --- 3A: Indigenous % trend by care type ---
    if 'pct_indigenous' in demo.columns:
        ind_trend = (
            demo.groupby(['YEAR', 'CARE_TYPE'])['pct_indigenous']
            .mean()
            .reset_index()
        )
        print('\nIndigenous % in admissions by year and care type:')
        print(ind_trend.pivot(index='YEAR', columns='CARE_TYPE', values='pct_indigenous')
              .round(3).to_string())
        print('\n>> If residential > home care: Indigenous people disproportionately')
        print('   in institutional care, suggesting home care access barriers.')
        print('   If both are LOW relative to Indigenous population share (~3.8% national):')
        print('   either undercount in data or genuinely lower aged care participation.')

    # --- 3B: CALD % trend ---
    if 'pct_cald' in demo.columns:
        cald_trend = (
            demo.groupby(['YEAR', 'CARE_TYPE'])['pct_cald']
            .mean()
            .reset_index()
        )
        print('\nCALD (non-English) % in admissions by year and care type:')
        print(cald_trend.pivot(index='YEAR', columns='CARE_TYPE', values='pct_cald')
              .round(3).to_string())

    # --- 3C: Top 10 ACPR regions by Indigenous % (most recent year) ---
    if 'pct_indigenous' in demo.columns:
        latest_year = demo['YEAR'].max()
        top_ind_acpr = (
            demo[demo['YEAR'] == latest_year]
            .groupby(['ACPR_NAME_2018', 'STATE'])['pct_indigenous']
            .mean()
            .sort_values(ascending=False)
            .head(10)
        )
        print(f'\nTop 10 ACPR regions by Indigenous % in admissions ({latest_year}):')
        print(top_ind_acpr.round(3).to_string())

## Key Findings Summary

*(Fill in after running all cells above — these become the dashboard annotations)*

---

### Chapter 1 — Did the Reform Work?
- [ ] Quality score direction after Oct 2023: **up / flat / down**
- [ ] Remote vs metro improvement gap: remote improved **more / less / same**
- [ ] Surprising class (if any):
- **Headline:** "The Oct 2023 mandate ___"

---

### Chapter 2 — Collision Zones
- [ ] Number of access-desert SA3s (below median on both):
- [ ] State with most access-desert SA3s:
- [ ] Worst care_gap_index SA3 name:
- **Headline:** "___ SA3 regions are caught with both low supply and low quality"

---

### Chapter 3 — Who Is Left Behind?
- [ ] Indigenous % in residential vs home care gap:
- [ ] CALD trend (rising / stable / falling):
- [ ] Age at entry trend:
- **Headline:** "The people most at risk are ___"

---

### Dashboard narrative arc (one sentence per chapter to use as slide text)
1. **Reform:** 
2. **Collision:** 
3. **People:** 
4. **Reveal:** "These are the communities Australia's aged care system has failed most."